# Day 6: Data Quality, Validation & Quarantine DLQ Engine

This notebook simulates running validation on intentionally bad records, verifying they are routed to a quarantine DLQ.

In [ ]:
import datetime
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, current_timestamp

spark = SparkSession.builder.appName("Day06_DQ_Demo").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

In [ ]:
# Create Mock Data with intentional errors
data = [
    ("P1", datetime.date(1980, 1, 1), 75),   # Valid
    (None, datetime.date(1990, 5, 5), 80),   # Invalid: Null ID
    ("P3", datetime.date(2050, 1, 1), 60),   # Invalid: Future birth date
    ("P4", datetime.date(1975, 8, 20), 500)  # Invalid: Clinically implausible heart rate
]
schema = ["person_id", "birth_date", "heart_rate"]
df_input = spark.createDataFrame(data, schema)

print("INPUT DATA:")
df_input.show()

In [ ]:
# Apply DQ Rules
df_evaluated = df_input \
    .withColumn("err_null_id", col("person_id").isNull()) \
    .withColumn("err_future_date", col("birth_date") > current_timestamp().cast("date")) \
    .withColumn("err_invalid_hr", (col("heart_rate") < 0) | (col("heart_rate") > 300)) \
    .withColumn("is_valid", ~(col("err_null_id") | col("err_future_date") | col("err_invalid_hr")))

df_valid = df_evaluated.filter(col("is_valid")).drop("err_null_id", "err_future_date", "err_invalid_hr", "is_valid")

# Create DLQ manifest
df_dlq = df_evaluated.filter(~col("is_valid")) \
    .withColumn("dlq_timestamp", current_timestamp()) \
    .withColumn("error_manifest", 
                lit("Failed Validation Rules. See boolean flags for details."))

print("VALID RECORDS (Routed to Silver):")
df_valid.show()

print("QUARANTINED RECORDS (Routed to DLQ):")
df_dlq.show(truncate=False)